# Mantenimiento Predictivo con NASA CMAPSS

Este notebook implementa un pipeline completo de **mantenimiento predictivo** usando el dataset CMAPSS (Commercial Modular Aero-Propulsion System Simulation) de la NASA.

## Dataset
- **Archivo**: `train_FD001.txt` y `RUL_FD001.txt`
- **Descripción**: Datos de sensores de motores de avión hasta el fallo.
- **Descarga**: [NASA Prognostics Center](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/)

## Estructura del notebook
1. Imports y configuración
2. Funciones auxiliares
3. Carga y preprocesamiento de datos
4. Análisis exploratorio con PCA
5. Modelado de series de tiempo (ARIMA)
6. Detección de anomalías (Isolation Forest)
7. Clasificación con Random Forest
8. Clasificación con LSTM
9. LSTM + Atención
10. Comparación por motor

## 1. Imports y configuración

En Python, una biblioteca (o módulo) es un conjunto de funciones ya escritas que se cargan al inicio del script. Es equivalente a cargar un bloque de función en un PLC o abrir un archivo de Excel con macros predefinidas: no se programa desde cero, se reutiliza trabajo existente.

Las bibliotecas que se cargan aquí son:

- **pandas**: maneja tablas de datos de forma programática, con operaciones equivalentes a filtros, agrupaciones y transformaciones de columnas en Excel, pero sin bucles manuales.
- **numpy**: realiza operaciones matemáticas sobre arreglos numéricos. Es la base del cómputo numérico en Python; pandas y sklearn lo usan internamente.
- **matplotlib**: genera gráficas estáticas (histogramas, series de tiempo, nubes de puntos).
- **sklearn** (scikit-learn): biblioteca de machine learning con implementaciones listas de preprocesamiento, reducción de dimensionalidad, modelos de clasificación y métricas de evaluación.
- **statsmodels**: modelos estadísticos clásicos, en particular modelos de series de tiempo como ARIMA.
- **tensorflow / keras**: framework para construir y entrenar redes neuronales. Keras es la interfaz de alto nivel sobre TensorFlow que permite definir arquitecturas de redes con pocas líneas de código.

La instrucción `seed=42` fija el estado del generador de números aleatorios de numpy y TensorFlow. Esto garantiza que cualquier proceso que involucre aleatoriedad (inicialización de pesos de redes, partición de datos) produzca los mismos resultados en cada ejecución, lo cual es necesario para que los experimentos sean reproducibles.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, f1_score

import statsmodels.api as sm

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Input, Layer
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 2. Funciones auxiliares

### 2.1 Helpers de visualización

Antes de construir los modelos se definen tres funciones de graficación que se usarán repetidamente. Definirlas aquí, en lugar de escribir el código de graficación cada vez, tiene la misma lógica que crear un subprograma en un PLC: se escribe una vez y se llama con distintos argumentos.

- `plot_hist(data, bins, title, xlabel, ylabel)`: genera un histograma. Un histograma muestra cuántas veces aparece cada valor (o rango de valores) en un conjunto de datos. Es la forma más directa de ver la distribución de una variable, por ejemplo cuántos ciclos tienen un determinado valor de RUL.
- `plot_line(y, title, xlabel, ylabel, x)`: genera una serie temporal o curva continua sobre un eje de ciclos u otro índice. Se usa para ver cómo evoluciona una señal a lo largo del tiempo.
- `plot_two_lines(...)`: superpone dos curvas en la misma gráfica para comparar su comportamiento directamente. Admite una línea horizontal de umbral (`hline`) para marcar un nivel de referencia.

In [ ]:
def plot_hist(data, bins, title, xlabel, ylabel):
    plt.figure(figsize=(8, 4))
    plt.hist(data, bins=bins, alpha=0.7)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()


def plot_line(y, title, xlabel, ylabel, x=None):
    plt.figure(figsize=(8, 4))
    if x is None:
        plt.plot(y, alpha=0.8)
    else:
        plt.plot(x, y, alpha=0.8)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()


def plot_two_lines(x, y1, y2, labels, title, xlabel, ylabel, styles=None, hline=None):
    plt.figure(figsize=(10, 4))
    style1 = '-'  if not styles else styles[0]
    style2 = '--' if not styles else styles[1]
    plt.plot(x, y1, label=labels[0], linestyle=style1)
    plt.plot(x, y2, label=labels[1], linestyle=style2)
    if hline is not None:
        plt.axhline(hline, color='red', linestyle=':', linewidth=0.8, label='Threshold')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.show()

### 2.2 Construcción de secuencias y splits

Esta sección define las funciones más importantes del notebook desde el punto de vista metodológico.

**¿Qué es una ventana deslizante?**

Un modelo de series de tiempo necesita aprender de secuencias de valores consecutivos, no de puntos aislados. `make_sequences` convierte la tabla de datos en pares (secuencia de entrada, etiqueta de salida): toma los 30 ciclos anteriores a cada instante como entrada y la etiqueta de distress del ciclo siguiente como salida. La ventana se mueve un ciclo a la vez a lo largo de la historia del motor. Es análogo al cálculo de una media móvil en un historiador de proceso, donde la ventana avanza punto a punto.

- `make_sequences`: ventana univariada (un sensor) → insumo para Random Forest
- `make_lstm_sequences`: ventana multivariada (varios sensores) → cada ventana tiene forma `(30 ciclos × n_sensores)`, insumo para LSTM
- `unit_time_series_splits`: divide los motores en conjuntos de entrenamiento y prueba respetando el orden cronológico. Esto es fundamental: si se permitiera que datos "futuros" informaran el entrenamiento, el modelo aprendería a ver el futuro y sus métricas no reflejarían el desempeño real en operación. Es análogo a calibrar un modelo de pronóstico usando eventos que aún no habían ocurrido cuando se tomó la decisión.
- `compute_metrics`: calcula cuatro métricas de desempeño del clasificador dado un vector de probabilidades predichas y las etiquetas reales. Maneja el caso en que un fold no contenga ambas clases (devuelve `NaN` en lugar de error).

In [ ]:
def make_sequences(df, sensor, window=30):
    """Construye ventanas deslizantes univariadas para un sensor dado."""
    sequences, labels, units = [], [], []
    for unit_id, group in df.groupby('unit'):
        values  = group[sensor].values
        targets = group['distress'].values
        for i in range(len(values) - window):
            sequences.append(values[i:i + window])
            labels.append(targets[i + window])
            units.append(unit_id)
    return np.array(sequences), np.array(labels), np.array(units)


def make_lstm_sequences(df, sensor_cols, window=30):
    """Construye ventanas deslizantes multivariadas para LSTM."""
    sequences, labels, units = [], [], []
    for unit_id, group in df.groupby('unit'):
        values  = group[sensor_cols].values
        targets = group['distress'].values
        for i in range(len(values) - window):
            sequences.append(values[i:i + window])
            labels.append(targets[i + window])
            units.append(unit_id)
    return np.array(sequences), np.array(labels), np.array(units)


def unit_time_series_splits(units: np.ndarray, n_splits: int = 5):
    """
    Divide cronológicamente por unidad (motor).
    El fold k usa los motores más tempranos para train y el siguiente bloque para test.
    """
    unique_units = np.array(sorted(pd.unique(units)))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    for train_u_idx, test_u_idx in tscv.split(unique_units):
        train_units = set(unique_units[train_u_idx])
        test_units  = set(unique_units[test_u_idx])
        train_idx = np.where(np.isin(units, list(train_units)))[0]
        test_idx  = np.where(np.isin(units, list(test_units)))[0]
        yield train_idx, test_idx


def compute_metrics(y_true, y_prob):
    """Calcula accuracy, ROC-AUC, PR-AUC y F1 para un clasificador binario."""
    y_hat = (y_prob >= 0.5).astype(int)
    return {
        'accuracy': float(accuracy_score(y_true, y_hat)),
        'roc_auc':  float(roc_auc_score(y_true, y_prob))
                    if len(np.unique(y_true)) > 1 else np.nan,
        'pr_auc':   float(average_precision_score(y_true, y_prob))
                    if len(np.unique(y_true)) > 1 else np.nan,
        'f1':       float(f1_score(y_true, y_hat))
                    if len(np.unique(y_hat)) > 1 else np.nan,
    }

## 3. Carga y preprocesamiento de datos

El archivo `test_FD001.txt` contiene lecturas de sensores de 100 motores de turbofán simulados con el software C-MAPSS de la NASA. Cada fila corresponde a un ciclo operacional de un motor. Las columnas son:

- **unit**: identificador del motor (1 a 100). Cada motor es una unidad independiente que opera desde un estado saludable hasta el fallo.
- **time**: número de ciclo dentro de la vida de ese motor. El ciclo 1 es el inicio de operación; el último ciclo registrado corresponde al fallo.
- **op_setting_1/2/3**: condiciones de operación en ese vuelo (altitud, número de Mach, temperatura ambiente). En el subconjunto FD001 estas condiciones son prácticamente constantes.
- **sensor_1 … sensor_21**: lecturas de 21 sensores que incluyen temperaturas, presiones, velocidades de rotor y relaciones de flujo en distintas etapas del motor.

**Cálculo del RUL y la etiqueta de distress**

A partir de los datos se calcula el RUL (Remaining Useful Life, Vida Útil Restante) de cada observación: para cada motor, el RUL en el ciclo `t` es `max_ciclo_del_motor - t`. Un motor con 200 ciclos de vida total tiene RUL = 200 al inicio y RUL = 0 en su último ciclo.

La etiqueta binaria **distress** es `True` cuando RUL < 20, es decir, cuando el motor está a menos de 20 ciclos del fallo. Este umbral convierte el problema de estimación continua de RUL en un problema de clasificación binaria: el modelo debe detectar cuándo un motor entra en esta zona de riesgo.

In [ ]:
df = pd.read_csv("data/CMaps/test_FD001.txt", sep=r"\s+", header=None)
df.dropna(axis=1, inplace=True)
df.columns = (
    ['unit', 'time', 'op_setting_1', 'op_setting_2', 'op_setting_3']
    + [f'sensor_{i}' for i in range(1, 22)]
)

# RUL ground-truth (usado como referencia; no se incluye en features)
rul = pd.read_csv("data/CMaps/RUL_FD001.txt", header=None)
rul.columns = ['max_RUL']
rul['unit'] = rul.index + 1

# Calcular RUL dentro del set de entrenamiento
last_cycle = df.groupby('unit')['time'].max().reset_index()
last_cycle.columns = ['unit', 'max_time']
df = df.merge(last_cycle, on='unit')
df['RUL']     = df['max_time'] - df['time']
df['distress'] = df['RUL'] < 20

print(f"Filas: {len(df):,}  |  Motores: {df['unit'].nunique()}")
print(f"Proporción en distress: {df['distress'].mean():.2%}")
df.head()

### Selección de sensores y distribución de RUL

**¿Por qué no usar los 21 sensores?**

No todos los sensores cambian de forma informativa conforme el motor se degrada. Algunos permanecen casi constantes durante toda la vida útil porque miden variables que no son sensibles al tipo de falla simulada (degradación del compresor de alta presión), o porque el sistema de control los regula activamente. Incluir señales sin variabilidad solo añade ruido al modelo sin aportar información.

La selección de `sensor_2, 3, 4, 9, 14, 17` se basa en análisis reportados en la literatura de CMAPSS: son los que muestran una tendencia sistemática de cambio conforme avanza la degradación. En el contexto de proceso, es análogo a seleccionar las señales que cambian durante una falla en un P&ID y descartar las que permanecen estables.

**¿Qué muestra el histograma de RUL?**

El histograma de RUL muestra cuántos ciclos del conjunto de datos tienen cada valor de vida restante. Si la distribución se concentra en valores altos de RUL, significa que la mayor parte de las observaciones provienen de motores en etapas tempranas de operación, lo cual es esperable: la etapa saludable dura mucho más que la etapa de degradación terminal. Esto también implica que los datos están desbalanceados: hay muchas más observaciones de clase "saludable" que de clase "distress".

## 4. Análisis con PCA

**¿Qué es PCA?**

El Análisis de Componentes Principales (PCA, por sus siglas en inglés) es una transformación matemática que convierte un conjunto de variables correlacionadas en un conjunto menor de variables nuevas llamadas componentes principales. Cada componente es una combinación lineal de los sensores originales, construida de forma que capture la mayor variabilidad posible en los datos.

En términos prácticos: si varios sensores tienden a moverse juntos cuando el motor se degrada, PCA los condensa en una o dos variables que resumen ese movimiento conjunto. Esto tiene dos ventajas: permite visualizar datos de alta dimensión en dos o tres dimensiones, y reduce el ruido al descartar componentes de variabilidad pequeña.

Antes de aplicar PCA se normaliza cada sensor con `StandardScaler`, que transforma los valores para que tengan media cero y desviación estándar uno. Esto es necesario porque los sensores tienen distintas unidades y rangos; sin normalización, un sensor con valores en miles dominaría sobre uno con valores en décimas.

**¿Por qué ajustar el scaler y la PCA solo sobre datos saludables (primeros 30 ciclos)?**

Si se entrenaran sobre el conjunto completo (incluidos ciclos de degradación y falla), los ejes de la PCA capturarían la variación de la degradación como si fuera comportamiento normal. Al ajustar solo con los primeros 30 ciclos, la PCA define el "espacio normal" de operación. Cualquier desviación observable en ese espacio es, por definición, anómala. Es el mismo principio que usar solo datos de operación normal para establecer los límites de control en una carta de Shewhart.

## 4. Análisis con PCA

Reducimos la dimensionalidad de los sensores a 3 componentes principales.
El scaler y la PCA se ajustan **solo sobre datos saludables** (primeros 30 ciclos)
para evitar que el deterioro influya en la definición de la zona normal.

In [ ]:
# Sensores más informativos para degradación del HPC
selected_sensors = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_9', 'sensor_14', 'sensor_17']

# Ajustar scaler y PCA solo sobre datos saludables (primeros 30 ciclos de vida)
# Nunca usar datos de degradación para definir la zona "normal"
healthy = df[df['time'] <= 30]
scaler_pca = StandardScaler()
X_healthy_sc = scaler_pca.fit_transform(healthy[selected_sensors])

pca = PCA(n_components=3)
pca.fit(X_healthy_sc)

print(f"Varianza explicada por componente: {pca.explained_variance_ratio_.round(3)}")
print(f"Varianza total (3 componentes)   : {pca.explained_variance_ratio_.sum():.2%}")

# Transformar todos los datos con el scaler y PCA ajustados sobre datos saludables
X_all_sc = scaler_pca.transform(df[selected_sensors])
pca_coords = pca.transform(X_all_sc)

df['pca_1'] = pca_coords[:, 0]
df['pca_2'] = pca_coords[:, 1]
df['pca_3'] = pca_coords[:, 2]

# Distancia euclidiana al centroide del espacio saludable (indicador de salud)
centroid = pca.transform(X_healthy_sc).mean(axis=0)
df['pca_distance'] = np.linalg.norm(pca_coords - centroid, axis=1)

# Umbral de anomalía: media + 3σ calculados sobre los ciclos saludables
dist_saludable = df[df['time'] <= 30]['pca_distance']
threshold = dist_saludable.mean() + 3 * dist_saludable.std()

print(f"\nCentroide PCA (espacio saludable): {centroid.round(3)}")
print(f"Umbral de anomalía (μ + 3σ)      : {threshold:.4f}")

### Visualización de la zona de operación saludable en el espacio PCA

La gráfica muestra una nube de puntos donde cada punto es un ciclo de operación de algún motor durante sus primeros 30 ciclos de vida. Los ejes PC1 y PC2 no corresponden a ningún sensor físico individual: son las dos direcciones de mayor variabilidad en el espacio de los seis sensores seleccionados.

Esta nube define el "envolvente de operación saludable" del motor. La idea es análoga a un diagrama de control estadístico de proceso (SPC): la región donde se concentran los puntos durante operación normal establece el comportamiento esperado. Si en ciclos posteriores un motor genera puntos fuera de esta región, es una señal de que sus lecturas de sensor se han alejado del estado saludable, lo que puede indicar degradación.

### Detección de anomalías por distancia al centroide PCA

**Concepto de distancia como indicador de salud**

Si un motor saludable produce lecturas que se agrupan en una región del espacio PCA, entonces la distancia de un punto nuevo al centro de esa región es un indicador de qué tan normal es ese ciclo: un punto muy alejado del centro refleja lecturas inusuales, posiblemente asociadas a degradación.

Se calcula la distancia euclidiana de cada observación al centroide (promedio) de la nube saludable en el plano PC1-PC2. Esta es una aproximación a la distancia de Mahalanobis, que además consideraría la forma (elipsoide) de la nube, no solo su centro.

**El umbral de detección**

El umbral se fija en media + 3σ de las distancias observadas en el conjunto completo. Esta regla proviene de la estadística: en una distribución aproximadamente normal, el 99.7% de los valores caen dentro de ±3σ respecto a la media. Las observaciones que exceden ese umbral son estadísticamente inusuales. Es el mismo criterio que los límites de control de 3σ en las cartas de Shewhart en manufactura: no implica necesariamente falla, pero sí una señal de que algo ha cambiado.

## 5. Modelado de series de tiempo con ARIMA

**¿Qué es ARIMA?**

ARIMA (AutoRegressive Integrated Moving Average) es un modelo estadístico para series de tiempo. Se compone de tres partes:

- **AR (p)** — *Autoregresivo*: el valor actual depende de los `p` valores anteriores. Equivale a decir "lo que pasó en los últimos `p` ciclos explica parcialmente lo que pasa hoy".
- **I (d)** — *Integrado*: se aplican `d` diferenciaciones para hacer la serie estacionaria (que su media y varianza no cambien con el tiempo). `d=1` equivale a modelar el cambio entre ciclos consecutivos en lugar del valor absoluto.
- **MA (q)** — *Media móvil*: el valor actual también depende de los `q` errores de predicción anteriores.

**Analogía industrial**: es como un controlador PID que usa el historial de error y su derivada para predecir el siguiente estado del proceso.

**¿Por qué ARIMA para mantenimiento predictivo?**

ARIMA ajusta un modelo de "comportamiento normal" sobre los datos históricos. Cuando el sensor empieza a degradarse, los valores reales se desvían del comportamiento esperado por el modelo — esos desvíos son los **residuales**. Un residual anormalmente grande indica que algo cambió en el sistema.

In [ ]:
# Nube de operación saludable en espacio PCA
healthy_df = df[df['time'] <= 30]
X_pca_healthy = healthy_df[['pca_1', 'pca_2']].values

plt.figure(figsize=(7, 5))
plt.scatter(X_pca_healthy[:, 0], X_pca_healthy[:, 1], alpha=0.3, s=10)
plt.title('PCA — Zona de operación saludable (primeros 30 ciclos)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

### Análisis de residuos del modelo ARIMA

**¿Qué es un residuo?**

El residuo en cada ciclo es la diferencia entre lo que el motor realmente midió y lo que el modelo ARIMA predijo para ese ciclo. Si el modelo captura bien la dinámica de la señal y el motor opera normalmente, los residuos son pequeños y parecen ruido aleatorio sin patrón.

Las bandas rojas en ±2σ funcionan como los límites de control en una carta de Shewhart: mientras los residuos se mantengan dentro de esas bandas, el proceso es estadísticamente estable. Excursiones frecuentes o sostenidas fuera de esos límites son evidencia de que la dinámica del sistema cambió: el modelo ya no puede explicar el comportamiento de la señal, lo que en el contexto de un motor en degradación indica que el daño ha alcanzado un nivel que altera la firma de los sensores de forma detectable.

## 6. Detección de anomalías — Isolation Forest

**¿Cómo funciona Isolation Forest?**

Isolation Forest es un algoritmo de detección de anomalías no supervisado: no requiere etiquetas de falla durante el entrenamiento. Se basa en la observación de que las anomalías son raras y diferentes del resto, por lo que son fáciles de aislar con pocas particiones aleatorias del espacio de datos.

El algoritmo construye muchos árboles de decisión con cortes aleatorios. Para cada punto, mide cuántos cortes se necesitan para aislarlo completamente. Los puntos que se aíslan con pocos cortes son candidatos a anomalías.

**Analogía industrial**: es como inspeccionar piezas en una línea de producción. Las piezas defectuosas se detectan rápido porque son fácilmente distinguibles del lote normal.

**Parámetro `contamination`**: fracción esperada de anomalías en los datos de entrenamiento. Se fija en 0.05 (5%), lo que calibra el umbral de decisión del modelo.

## 5. Modelado de series de tiempo con ARIMA

Ajustamos un ARIMA(1,1,1) sobre `pca_1` del motor 1 y analizamos
los residuos para detectar comportamiento anómalo.

In [ ]:
# ARIMA(1,1,1) sobre la PC1 del motor 1
# d=1 porque la serie de pca_1 tiene tendencia (no estacionaria)
motor_1_pca1 = df[df['unit'] == 1]['pca_1'].values

arima_model = sm.tsa.ARIMA(motor_1_pca1, order=(1, 1, 1))
result = arima_model.fit()

residuals = result.resid

print(result.summary())
print(f"\nDesvío estándar de residuos : {residuals.std():.4f}")
print(f"Ciclos anómalos (|r| > 2σ)  : {(np.abs(residuals) > 2 * residuals.std()).sum()}")

### Distribución de distancias PCA y umbral de anomalía

El histograma muestra cuántos ciclos de todos los motores tienen cada valor de distancia al centroide saludable. La distribución debería mostrar un cuerpo principal concentrado en valores bajos (ciclos de operación normal) con una cola larga hacia la derecha (ciclos en etapas de degradación y falla).

La línea vertical roja es el umbral calculado (media + 3σ). Las observaciones a su derecha se clasifican como anómalas. Este gráfico permite verificar que el umbral está razonablemente posicionado: si cae dentro del cuerpo principal de la distribución, está demasiado bajo y generará exceso de falsas alarmas; si cae demasiado a la derecha en la cola, dejará pasar sin detectar muchos eventos de degradación real.

## 7. Clasificación de secuencias — Random Forest

**¿Qué es un árbol de decisión y por qué un bosque?**

Un árbol de decisión es una secuencia de preguntas binarias aplicadas a las variables de entrada, que termina en una clasificación. Por ejemplo: "¿El sensor_9 en el ciclo 28 está por encima de X? → Sí → ¿El sensor_9 en el ciclo 27 bajó más de Y? → No → Clasificar como Normal."

Un árbol solo puede ser tan complejo como sus datos de entrenamiento lo permitan, y tiende a sobreajustarse. Un **Random Forest** entrena muchos árboles, cada uno con una muestra aleatoria de los datos y un subconjunto aleatorio de las variables. La predicción final es la votación mayoritaria de todos los árboles.

**¿Por qué ventanas deslizantes?**

El clasificador no recibe la serie de tiempo directamente — recibe una ventana de 30 ciclos consecutivos como un vector de características. Cada posición en la ventana es una variable de entrada. Esto convierte el problema de series de tiempo en un problema de clasificación tabular estándar.

In [ ]:
sigma = residuals.std()

plt.figure(figsize=(10, 4))
plt.plot(residuals, alpha=0.7, label='Residuos')
plt.axhline( 2 * sigma, color='r', linestyle='--', label='+2σ')
plt.axhline(-2 * sigma, color='r', linestyle='--', label='-2σ')
plt.title('Residuos ARIMA sobre PC1 — Motor 1')
plt.xlabel('Ciclo')
plt.ylabel('Residuo')
plt.legend()
plt.tight_layout()
plt.show()

### Entrenamiento con validación cruzada cronológica por motor

**¿Qué es la validación cruzada y por qué es necesaria?**

Si se entrenara el modelo con todos los datos disponibles y luego se midiera su desempeño sobre esos mismos datos, el resultado sería artificialmente optimista: el modelo ya vio esos datos y los "memorizó". La validación cruzada evalúa el modelo sobre datos que no usó durante el entrenamiento.

Aquí se usan 5 folds cronológicos por motor: en el fold 1, los primeros motores (por ID numérico) se usan para entrenar y los siguientes para probar. En el fold 2, el grupo de entrenamiento se amplía y el de prueba cambia. Este esquema simula el escenario realista en que el modelo se entrena con datos históricos de una flota y se evalúa en motores que entraron en servicio después.

En cada fold:
1. Se separan las secuencias según el grupo de motores asignado a cada partición
2. Se entrena un Random Forest de 100 árboles
3. Se calculan probabilidades de distress para el conjunto de prueba
4. Se evalúan las métricas

El promedio de las métricas sobre los 5 folds es la estimación del desempeño del modelo en condiciones de operación real.

### Distribución de probabilidades predichas — último fold

El Random Forest no produce una clasificación binaria directa, sino una probabilidad entre 0 y 1 de que la observación esté en distress (basada en la fracción de árboles que votaron por esa clase). El umbral estándar para convertir probabilidad en clase es 0.5.

El histograma muestra cómo se distribuyen esas probabilidades en el conjunto de prueba del último fold. Un modelo que discrimina bien produce una distribución bimodal: muchas observaciones con probabilidad cercana a 0 (claramente saludables) y muchas con probabilidad cercana a 1 (claramente en distress), con pocos valores en la zona intermedia donde el modelo es incierto.

Si la distribución es plana o se concentra en valores intermedios, el modelo no distingue con certeza entre las dos clases, lo que en la práctica produciría muchas alarmas dudosas o falsas alarmas.

## 8. Clasificación con LSTM

**¿Qué es una red LSTM?**

Una red LSTM (Long Short-Term Memory, Memoria de Largo y Corto Plazo) es un tipo de red neuronal diseñada específicamente para aprender patrones en secuencias temporales. A diferencia de una red neuronal convencional, que procesa cada entrada de forma independiente, una LSTM mantiene un estado interno (una "memoria") que se actualiza en cada paso de tiempo. Esto le permite capturar relaciones entre ciclos separados en el tiempo: si el deterioro visible en el ciclo 50 está relacionado con el comportamiento del ciclo 20, la LSTM puede aprender esa relación.

En el contexto de señales industriales, es análoga a un estimador de estado que combina observaciones pasadas con el estado estimado anterior para producir una estimación del estado actual. La diferencia es que los parámetros de esa combinación se aprenden automáticamente de los datos, sin necesidad de un modelo físico del sistema.

**Arquitectura de este modelo**

- **Entrada**: ventana de 30 ciclos × 6 sensores, transformada a 3 componentes PCA dentro de cada fold
- **LSTM con 64 unidades**: procesa la secuencia paso a paso, actualizando su estado interno en cada ciclo
- **Capa densa con activación sigmoide**: convierte el estado final de la LSTM en una probabilidad entre 0 y 1

> El scaler y la PCA se ajustan siempre sobre los datos de entrenamiento de cada fold. Aplicarlos primero sobre el conjunto de prueba filtraría información de los datos de prueba hacia el modelo, lo que inflaría artificialmente las métricas.

In [ ]:
pca_features = df[['pca_1', 'pca_2', 'pca_3']].values

clf = IsolationForest(contamination=0.05, random_state=42)
df['anomaly_iforest'] = clf.fit_predict(pca_features) == -1

print(f"Anomalías Isolation Forest: {df['anomaly_iforest'].sum():,} ({df['anomaly_iforest'].mean():.2%})")

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(df['pca_distance'], bins=50, alpha=0.6, label='Distancia PCA')
plt.axvline(threshold, color='red', linestyle='--', label=f'Umbral ({threshold:.2f})')
plt.title('Distribución de distancia PCA al centroide saludable')
plt.xlabel('Distancia')
plt.ylabel('Frecuencia')
plt.legend()
plt.tight_layout()
plt.show()

### Evolución de la probabilidad de distress predicha por LSTM

La gráfica muestra la probabilidad de distress asignada por el modelo LSTM a cada ventana de 30 ciclos en el último fold, en el orden en que aparecen los datos. A diferencia del histograma anterior, aquí el eje horizontal preserva la secuencia, lo que permite observar el comportamiento temporal de las predicciones.

En un escenario ideal, las predicciones deberían arrancar cerca de 0 (motor saludable) e ir aumentando progresivamente hacia 1 conforme el motor se acerca al fallo. Fluctuaciones abruptas en etapas tempranas o predicciones elevadas sostenidas mucho antes del fallo indicarían que el modelo es demasiado conservador o que está reaccionando a ruido en lugar de a degradación real.

## 9. LSTM + Mecanismo de Atención

**¿Qué es el mecanismo de atención?**

En el LSTM estándar, el estado final de la red es el que se pasa a la capa de salida para hacer la predicción. Ese estado resume toda la secuencia de 30 ciclos, pero no distingue cuáles ciclos fueron más informativos para el diagnóstico actual.

El mecanismo de atención añade una capa que aprende a asignar un peso a cada paso de tiempo: algunos ciclos dentro de la ventana reciben más "atención" que otros para producir la predicción. El resultado es un vector de contexto que pondera las representaciones de cada ciclo por su relevancia aprendida.

La analogía industrial: un técnico que revisa 30 registros históricos antes de emitir un diagnóstico no les da el mismo peso a todos. Los picos anómalos recientes, o los registros justo antes de un evento, reciben más atención. El mecanismo de atención automatiza esa ponderación a partir de los datos.

**Ventaja adicional**

Los pesos de atención son interpretables: se puede visualizar en cuáles ciclos dentro de la ventana el modelo "miró más" para producir cada predicción. Esto añade transparencia al sistema de prognósticos, lo que en contextos industriales es importante para validar y justificar las alertas generadas.

In [ ]:
X_seq, y_seq, units_rf = make_sequences(df, 'sensor_9', window=30)
print(f"Secuencias: {X_seq.shape}  |  Positivos (distress): {y_seq.sum():,} ({y_seq.mean():.2%})")

In [ ]:
rf_metrics = []
y_pred_rf  = None  # probabilidades del último fold (para graficar)

for fold, (train_idx, test_idx) in enumerate(
    unit_time_series_splits(units_rf, n_splits=5), start=1
):
    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y_seq[train_idx], y_seq[test_idx]

    clf_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    clf_rf.fit(X_train, y_train)

    y_prob = clf_rf.predict_proba(X_test)[:, 1]
    metrics = compute_metrics(y_test, y_prob)
    rf_metrics.append(metrics)
    print(f"Fold {fold}: {metrics}")

    y_pred_rf = y_prob  # se sobreescribe en cada fold; al final contiene el último

rf_mean = {k: np.nanmean([m[k] for m in rf_metrics]) for k in rf_metrics[0]}
print(f"\nMedia RF (5 folds): {rf_mean}")

### Construcción del modelo LSTM con atención — API funcional de Keras

Se usa la API funcional de Keras (en lugar de `Sequential`) porque el modelo necesita producir dos salidas simultáneamente: la probabilidad de distress y los pesos de atención por paso de tiempo. Con `Sequential` solo se puede definir un flujo lineal con una sola salida, lo que no es suficiente aquí.

La arquitectura, capa por capa:

1. **`Input`**: declara la forma de los datos de entrada (30 ciclos × 3 componentes PCA)
2. **`LSTM(64, return_sequences=True)`**: en lugar de devolver solo el estado final, devuelve la representación oculta de cada uno de los 30 pasos de tiempo. Esto es necesario para que la capa de atención pueda ponderar cada ciclo individualmente.
3. **`AttentionWithWeights`**: calcula un peso para cada paso de tiempo (entre 0 y 1, sumando 1 en total), multiplica cada representación por su peso y suma para obtener un único vector de contexto. También devuelve los pesos para visualización posterior.
4. **`Dense(1, sigmoid)`**: convierte el vector de contexto en una probabilidad de distress entre 0 y 1.

### Entrenamiento del modelo con atención

El proceso de entrenamiento de una red neuronal consiste en presentar lotes de secuencias al modelo, calcular el error entre la predicción y la etiqueta real (función de pérdida), y ajustar los pesos internos de la red para reducir ese error. Esto se repite durante múltiples épocas (pasadas completas por los datos de entrenamiento).

Los callbacks configurados controlan cuándo detener el entrenamiento:

- **`EarlyStopping(patience=3)`**: si la pérdida sobre el conjunto de validación no mejora durante 3 épocas consecutivas, el entrenamiento se detiene y se restauran los pesos de la mejor época. Esto evita que el modelo memorice los datos de entrenamiento a costa de perder capacidad de generalización.
- **`ReduceLROnPlateau(patience=2, factor=0.5)`**: si el modelo se estanca (sin mejora durante 2 épocas), reduce la tasa de aprendizaje a la mitad. La tasa de aprendizaje controla qué tan grandes son los ajustes de pesos en cada paso; reducirla permite afinar la solución cuando ya se está cerca del óptimo.

El modelo se entrena sobre los datos del último fold para mantener coherencia con el análisis por motor que sigue.

## 10. Comparación por motor

Las métricas globales (ROC-AUC, F1, etc.) resumen el desempeño del modelo sobre toda la población de motores en el conjunto de prueba, pero no muestran cómo se comporta en un motor individual a lo largo de su vida. Esta sección construye la trayectoria completa de probabilidad de distress para motores seleccionados, aplicando los modelos ya entrenados ciclo a ciclo.

Lo que se espera observar en un motor que funciona correctamente: probabilidad baja y estable durante la mayor parte de la vida, con un incremento sostenido en los últimos ciclos antes del fallo.

In [ ]:
plot_hist(
    y_pred_rf, bins=30,
    title='Random Forest — Probabilidades de distress (último fold)',
    xlabel='Probabilidad predicha', ylabel='Frecuencia'
)

### Construcción de secuencias para el motor seleccionado

Se extraen todos los ciclos del motor `UNIT_ID = 3` y se construyen secuencias deslizantes de 30 ciclos con el mismo procedimiento que durante el entrenamiento. Se aplica el mismo `scaler` y la misma `PCA` ajustados sobre los datos de entrenamiento del último fold.

Este punto es importante: si se reajustaran el scaler o la PCA sobre los datos del motor individual, se estaría usando información de ese motor para normalizar, lo que produciría predicciones artificialmente buenas. En operación real, el modelo ya fue calibrado con datos históricos y se aplica tal cual a cada motor nuevo que entra al sistema de monitoreo.

### Comparación entre LSTM y LSTM+Atención para un motor específico

La gráfica superpone las probabilidades predichas por ambos modelos a lo largo de toda la vida del motor seleccionado. La línea punteada horizontal en 0.5 es el umbral de decisión: por encima, el modelo emite una alerta de distress.

Lo que se busca observar en esta comparación:

- **Anticipación de la alarma**: ¿cuántos ciclos antes del fallo cada modelo supera el umbral de 0.5? Una alerta más anticipada da más margen para planificar la intervención.
- **Estabilidad de la señal**: ¿la probabilidad aumenta de forma progresiva o hay fluctuaciones que generarían alarmas intermitentes (una alerta que aparece y desaparece varios ciclos antes del fallo)?
- **Efecto de la atención**: ¿el mecanismo de atención produce una señal más temprana, más tardía, o simplemente con diferente forma que el LSTM estándar?

En mantenimiento predictivo, el valor operacional del sistema depende de que las alertas sean tempranas, estables y con pocos falsos positivos. Una alerta tardía o intermitente reduce la utilidad del sistema para la toma de decisiones.

## 11. Resumen de métricas

La tabla compara el desempeño de los dos modelos supervisados promediado sobre los 5 folds de validación cruzada. Las métricas utilizadas son:

- **Accuracy**: fracción de predicciones correctas (tanto saludables como en distress correctamente clasificadas). Es intuitiva pero engañosa cuando las clases están desbalanceadas: si el 90% de los ciclos son saludables, un modelo que siempre predice "saludable" tendría 90% de accuracy sin detectar ningún fallo.

- **ROC-AUC** (Area Under the Receiver Operating Characteristic curve): mide la capacidad del modelo para ordenar correctamente las observaciones. Un valor de 1.0 significa que el modelo siempre asigna probabilidades más altas a los casos en distress que a los saludables, sin importar el umbral elegido. Un valor de 0.5 equivale a predicción aleatoria. Es útil porque no depende del umbral de clasificación.

- **PR-AUC** (Area Under the Precision-Recall curve): más informativa que ROC-AUC cuando las clases están desbalanceadas. La precisión mide qué fracción de las alertas emitidas son verdaderos fallos; el recall mide qué fracción de los fallos reales fueron detectados. Una PR-AUC alta indica que el modelo detecta la mayoría de los fallos con pocas falsas alarmas.

- **F1**: media armónica de precisión y recall. Resume en un solo número el balance entre detectar fallos reales y evitar falsas alarmas. Es cero si el modelo no detecta ningún fallo real o si todas sus alertas son falsas, y es uno si detecta todos los fallos sin ninguna falsa alarma.

In [ ]:
X, y, units_lstm = make_lstm_sequences(df, selected_sensors, window=30)
n_features = len(selected_sensors)
print(f"Secuencias LSTM: {X.shape}")

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=0),
]

lstm_metrics = []

for fold, (train_idx, test_idx) in enumerate(
    unit_time_series_splits(units_lstm, n_splits=5), start=1
):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Normalización y PCA: ajustar SOLO en train
    scaler_lstm = StandardScaler().fit(X_train.reshape(-1, n_features))
    X_train_sc  = scaler_lstm.transform(X_train.reshape(-1, n_features)).reshape(X_train.shape)
    X_test_sc   = scaler_lstm.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)

    pca_lstm   = PCA(n_components=3).fit(X_train_sc.reshape(-1, n_features))
    X_train_seq = pca_lstm.transform(X_train_sc.reshape(-1, n_features)).reshape(X_train.shape[0], X_train.shape[1], 3)
    X_test_seq  = pca_lstm.transform(X_test_sc.reshape(-1, n_features)).reshape(X_test.shape[0], X_test.shape[1], 3)

    # Modelo LSTM
    model_lstm = Sequential([
        LSTM(64, input_shape=(X_train_seq.shape[1], 3)),
        Dense(1, activation='sigmoid')
    ])
    model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model_lstm.fit(
        X_train_seq, y_train,
        epochs=20, batch_size=32, shuffle=False,
        validation_data=(X_test_seq, y_test),
        callbacks=callbacks, verbose=0
    )

    y_pred_lstm = model_lstm.predict(X_test_seq, verbose=0).flatten()
    metrics = compute_metrics(y_test, y_pred_lstm)
    lstm_metrics.append(metrics)
    print(f"Fold {fold}: {metrics}")

lstm_mean = {k: np.nanmean([m[k] for m in lstm_metrics]) for k in lstm_metrics[0]}
print(f"\nMedia LSTM: {lstm_mean}")

In [ ]:
plot_line(
    y_pred_lstm,
    title='LSTM — Probabilidades de distress (último fold)',
    xlabel='Índice de muestra', ylabel='Probabilidad predicha'
)

## 9. LSTM + Mecanismo de Atención

El mecanismo de atención aprende a **ponderar los pasos de tiempo** más
relevantes para la predicción de distress, añadiendo interpretabilidad.

In [ ]:
class AttentionWithWeights(Layer):
    """Capa de atención con pesos observables para visualización."""

    def build(self, input_shape):
        self.W = self.add_weight(
            name='attn_weight',
            shape=(input_shape[-1], 1),
            initializer='random_normal',
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        scores  = tf.matmul(inputs, self.W)          # (batch, timesteps, 1)
        weights = tf.nn.softmax(scores, axis=1)      # normalización temporal
        context = tf.reduce_sum(inputs * weights, axis=1)  # suma ponderada
        return context, weights

In [ ]:
# Entrenamos el modelo de atención sobre el último fold (para visualización consistente)
input_seq  = Input(shape=(X_train_seq.shape[1], 3))
lstm_out   = LSTM(64, return_sequences=True)(input_seq)
context, attn_weights = AttentionWithWeights()(lstm_out)
output     = Dense(1, activation='sigmoid', name='pred')(context)

model_attn = Model(inputs=input_seq, outputs={'pred': output, 'attn': attn_weights})
model_attn.compile(
    optimizer='adam',
    loss={'pred': 'binary_crossentropy'},
    metrics={'pred': 'accuracy'}
)
model_attn.summary()

In [ ]:
model_attn.fit(
    X_train_seq, {'pred': y_train},
    epochs=10, batch_size=32, shuffle=False,
    validation_data=(X_test_seq, {'pred': y_test}),
    callbacks=callbacks, verbose=1
)

## 10. Comparación por motor

Visualizamos las predicciones de LSTM y LSTM+Atención para un motor específico
a lo largo de toda su vida operacional.

In [ ]:
UNIT_ID = 3
WINDOW  = 30

engine_data = df[df['unit'] == UNIT_ID].copy()

# Construir secuencias del motor
X_engine_raw = np.array([
    engine_data[selected_sensors].values[i:i + WINDOW]
    for i in range(len(engine_data) - WINDOW)
])
X_engine_sc = scaler_lstm.transform(
    X_engine_raw.reshape(-1, n_features)
).reshape(X_engine_raw.shape)

X_engine = pca_lstm.transform(
    X_engine_sc.reshape(-1, n_features)
).reshape(X_engine_raw.shape[0], WINDOW, 3)

cycles = engine_data['time'].values[WINDOW:]
print(f"Motor {UNIT_ID}: {len(cycles)} puntos de predicción")

In [ ]:
pred_lstm_engine  = model_lstm.predict(X_engine, verbose=0).flatten()
pred_dict         = model_attn.predict(X_engine, verbose=0)
pred_attn_engine  = pred_dict['pred'].flatten()

# Gráficas individuales
plot_line(
    pred_lstm_engine,
    title=f'LSTM — Probabilidad de distress, Motor {UNIT_ID}',
    xlabel='Ciclo', ylabel='Probabilidad',
    x=cycles
)
plot_line(
    pred_attn_engine,
    title=f'LSTM + Atención — Probabilidad de distress, Motor {UNIT_ID}',
    xlabel='Ciclo', ylabel='Probabilidad',
    x=cycles
)

In [ ]:
plot_two_lines(
    x=cycles,
    y1=pred_lstm_engine,
    y2=pred_attn_engine,
    labels=['LSTM', 'LSTM + Atención'],
    title=f'Motor {UNIT_ID} — LSTM vs LSTM + Atención',
    xlabel='Ciclo', ylabel='Probabilidad de distress',
    styles=['-', '--'],
    hline=0.5
)

## 11. Resumen de métricas

Comparamos los dos modelos supervisados sobre los 5 folds.

In [ ]:
summary = pd.DataFrame({
    'Modelo': ['Random Forest', 'LSTM'],
    'Accuracy':  [rf_mean['accuracy'],  lstm_mean['accuracy']],
    'ROC-AUC':   [rf_mean['roc_auc'],   lstm_mean['roc_auc']],
    'PR-AUC':    [rf_mean['pr_auc'],    lstm_mean['pr_auc']],
    'F1':        [rf_mean['f1'],        lstm_mean['f1']],
})
summary.set_index('Modelo', inplace=True)
summary.round(4)